In [ ]:
# ============================================================
#  휴플 KG × ETRI 추론엔진 — 컨텍스트 주입 테스트
#
#  목적: KG에서 뽑은 안전 정보를 프롬프트에 넣었을 때
#        LLM이 안전 문구를 원문 그대로 유지하는지 검증한다.
#
#  ※ Neo4j 연결 없이 동작합니다. KG 조회 결과를 미리 문자열로
#     넣어두었으므로, API 응답 품질만 확인하면 됩니다.
# ============================================================
import requests, json, re

# 코랩 보안 비밀 사용 권장 (왼쪽 열쇠 아이콘 → JEJU_API_KEY 등록)
try:
    from google.colab import userdata
    API_KEY = userdata.get("JEJU_API_KEY")
except Exception:
    API_KEY = "여기에_발급받은_키"   # 공유 시 반드시 지울 것

URL = "https://jejuax.ngrok.app/api/agent/v1/responses"
HEADERS = {"Content-Type": "application/json", "X-API-Key": API_KEY}
print("설정 완료")

In [ ]:
# ============================================================
#  프롬프트 규칙 + 응답 검증기
# ============================================================
INSTRUCTION = """다음은 제주 안전관광 지식그래프에서 조회한 검증된 정보입니다.

[작성 규칙]
1. 아래 제공된 정보만 사용하고, 없는 사실을 추측하거나 지어내지 마십시오.
2. [주의] [안전안내] [법적제한] 으로 표시된 문장은 반드시 원문 그대로 인용하고
   요약하거나 바꿔 쓰지 마십시오. 안전에 직결되는 문구입니다.
3. 사고 건수, 사망자 수 등 수치를 언급하지 마십시오. 제공된 문구에도 없습니다.
4. 접근성이 "정보 미확보"인 항목은 "이용 불가"가 아니라 "확인이 필요합니다"로
   안내하십시오.
5. [확인필요] 표시가 있으면 "이용 가능"이라고 단정하지 말고, 반드시
   사전 문의가 필요하다는 점을 함께 안내하십시오.
6. 친절하고 정돈된 문장으로 답변하되, 과장하거나 불안을 조장하지 마십시오.
"""

FORBIDDEN = re.compile(r"사망|숨진|사상자|치사|익사|\d+\s*명이?\s*(?:숨|사망|다)")

def compose(question, context):
    return f"{INSTRUCTION}\n[조회 정보]\n{context}\n\n[사용자 질문]\n{question}\n"

def extract(resp):
    out=[]
    for it in resp.get("output", []):
        if it.get("type")=="message":
            for c in it.get("content", []):
                if c.get("type")=="output_text":
                    out.append(c.get("text",""))
    return "\n".join(out)

def norm(s):
    return re.sub(r"[^\w가-힣]", "", s)

def validate(answer, verbatim):
    issues=[]
    if FORBIDDEN.search(answer):
        issues.append("금지 표현(사망/사상자) 포함")
    for v in verbatim:
        key = norm(v)[:14]
        if key and key not in norm(answer):
            issues.append(f"안전 문구 누락/변형 → {v[:30]}…")
    if "금지되어 있습니다" in answer:
        issues.append("시행 전 규정을 현재 시제로 단정")
    return issues

def run(question, context, verbatim, label=""):
    prompt = compose(question, context)
    print("="*60); print(f"[{label}] {question}")
    print(f"프롬프트 {len(prompt)}자")
    r = requests.post(URL, headers=HEADERS, timeout=300,
                      json={"model":"jeju-tourism-agent","input":prompt,"stream":False})
    r.raise_for_status()
    data = r.json()
    ans = extract(data)
    print(f"토큰: in {data.get('usage',{}).get('input_tokens')} / out {data.get('usage',{}).get('output_tokens')}")
    print("-"*60); print(ans)
    print("-"*60)
    iss = validate(ans, verbatim)
    print("검증 통과" if not iss else "검증 실패:")
    for i in iss: print("   -", i)
    return ans, iss

print("함수 준비 완료")

In [ ]:
# ============================================================
#  CASE A — 접근성 조건 검색 (휠체어)
#  ※ 숙박·교통시설은 제외됨 (관광지 추천이 목적)
# ============================================================
CONTEXT_A = """■ 산방산탄산온천
  - 유형: 웰니스
  - 접근성: 완전 이용 가능 (검증완료)
  - 시설 이용 가능 / 활동 참여 미확인
  - [확인필요] 활동 참여 가능 여부는 확인되지 않았습니다. 보조 인력이 필요할 수 있으니 시설에 미리 문의해 주세요
  - 장애인화장실 있음
  - [주변] 주변 지역에 안전사고 이력이 있습니다

■ 아르떼뮤지엄제주
  - 유형: 전시시설
  - 접근성: 완전 이용 가능 (검증완료)
  - 장애인화장실 있음
  - 운영시간: 10:00~20:00 (입장마감 19:00)
  - [주의] 입구 경사로·내부 무단차이나 일부 전시장 휠체어·유아차 진입 불가, 혼잡시간 이동 불편(오감 p.19)

■ 제주항공우주박물관
  - 유형: 전시시설
  - 접근성: 완전 이용 가능 (검증완료)
  - 장애인화장실 있음
  - 운영시간: 09:00~18:00
  - 요금: 10000
  - [주변] 주변 지역에 안전사고 이력이 있습니다

■ 렛츠런파크제주
  - 유형: 액티비티
  - 접근성: 완전 이용 가능 (검증완료)
  - 시설 이용 가능 / 활동 참여 미확인
  - [확인필요] 활동 참여 가능 여부는 확인되지 않았습니다. 보조 인력이 필요할 수 있으니 시설에 미리 문의해 주세요
  - 장애인화장실 있음
  - 운영시간: 09:30~18:30
  - 요금: 2000
  - [주변] 주변 지역에 안전사고 이력이 있습니다

■ 제주민속촌
  - 유형: 문화유산
  - 접근성: 완전 이용 가능 (검증완료)
  - 장애인화장실 있음
  - 운영시간: 08:30~18:00(4/1~7/15,9/1~9/30)·08:30~18:30(7/16~8/31)·08:30~17:30(3월)·08:30~17:00(10~2월)
  - 요금: 15000
  - [주변] 주변 지역에 안전사고가 잦은 편입니다. 이동 중 주의해 주세요"""
VERBATIM_A = ["입구 경사로·내부 무단차이나 일부 전시장 휠체어·유아차 진입 불가, 혼잡시간 이동 불편(오감 p.19)"]

ans_a, iss_a = run("휠체어로 이용하기 편한 제주 관광지 추천해줘",
                   CONTEXT_A, VERBATIM_A, "CASE A")

In [ ]:
# ============================================================
#  CASE B — 안전 안내 + 법적 제한 (가장 중요)
# ============================================================
CONTEXT_B = """■ 월령포구
  - 유형: 해양
  - 접근성: 부분 이용 가능
  - [안전안내] 여름철 오후에 안전사고가 반복해서 보고된 곳입니다. 이곳은 어촌정주어항으로 지정된 어항구역입니다. 2027년 4월 22일부터 어촌·어항법에 따라 물놀이·다이빙이 금지되며 위반 시 과태료가 부과됩니다. 부두와 방파제 가장자리, 테트라포드 위는 미끄러지기 쉬우니 올라가지 말아 주세요.
  - [법적제한] 어촌정주어항으로 지정되어 2027-04-22부터 물놀이·다이빙이 금지됩니다
  - [주변] 주변 지역에 안전사고 이력이 있습니다

■ 구엄포구
  - 유형: 해양
  - 접근성: 부분 이용 가능
  - 장애인화장실 없음
  - [안전안내] 오후에 안전사고가 반복해서 보고된 곳입니다. 이곳은 어촌정주어항으로 지정된 어항구역입니다. 2027년 4월 22일부터 어촌·어항법에 따라 물놀이·다이빙이 금지되며 위반 시 과태료가 부과됩니다. 부두와 방파제 가장자리, 테트라포드 위는 미끄러지기 쉬우니 올라가지 말아 주세요.
  - [법적제한] 어촌정주어항으로 지정되어 2027-04-22부터 물놀이·다이빙이 금지됩니다
  - [주변] 주변 지역에 안전사고가 잦은 편입니다. 이동 중 주의해 주세요

■ 함덕해수욕장
  - 유형: 해양
  - 접근성: 부분 이용 가능
  - 장애인화장실 있음
  - [안전안내] 여름철 오후에 안전사고가 반복해서 보고된 곳입니다. 지정된 물놀이 구역과 안전요원 근무 시간대를 이용해 주세요.
  - [주변] 주변 지역에 안전사고가 잦은 편입니다. 이동 중 주의해 주세요

■ 김녕해수욕장
  - 유형: 해양
  - 접근성: 부분 이용 가능
  - 장애인화장실 있음
  - [안전안내] 여름철에 안전사고가 반복해서 보고된 곳입니다. 지정된 물놀이 구역과 안전요원 근무 시간대를 이용해 주세요.
  - [주변] 주변 지역에 안전사고 이력이 있습니다"""
VERBATIM_B = ["여름철 오후에 안전사고가 반복해서 보고된 곳입니다. 이곳은 어촌정주어항으로 지정된 어항구역입니다. 2027년 4월 22일부터 어촌·어항법에 따라 물놀이·다이빙이 금지되며 위반 시 과태료가 부과됩니다. 부두와 방파제 가장자리, 테트라포드 위는 미끄러지기 쉬우니 올라가지 말아 주세요.", "어촌정주어항으로 지정되어 2027-04-22부터 물놀이·다이빙이 금지됩니다", "오후에 안전사고가 반복해서 보고된 곳입니다. 이곳은 어촌정주어항으로 지정된 어항구역입니다. 2027년 4월 22일부터 어촌·어항법에 따라 물놀이·다이빙이 금지되며 위반 시 과태료가 부과됩니다. 부두와 방파제 가장자리, 테트라포드 위는 미끄러지기 쉬우니 올라가지 말아 주세요.", "어촌정주어항으로 지정되어 2027-04-22부터 물놀이·다이빙이 금지됩니다", "여름철 오후에 안전사고가 반복해서 보고된 곳입니다. 지정된 물놀이 구역과 안전요원 근무 시간대를 이용해 주세요.", "여름철에 안전사고가 반복해서 보고된 곳입니다. 지정된 물놀이 구역과 안전요원 근무 시간대를 이용해 주세요."]

ans_b, iss_b = run("제주에서 물놀이하기 좋은 바다 알려줘",
                   CONTEXT_B, VERBATIM_B, "CASE B")

In [ ]:
# ============================================================
#  CASE C — 시설접근 ≠ 활동참여 / 근거 수준 구분
#  확인 포인트:
#    ① "부분 이용 가능"을 "가능"으로 뭉개지 않는가
#    ② 계단·보호자 동반·휠체어 대여 없음을 빠뜨리지 않는가
#    ③ "현장 확인된 등급이 아님"을 생략하지 않는가
# ============================================================
CONTEXT_C = """■ 산방산탄산온천
  - 유형: 웰니스
  - 접근성: 부분 이용 가능 (검증완료)
  - 장애인화장실 있음
  - [주의] 탕 내부에 계단이 있어 휠체어 단독 이용은 위험합니다. 보호자 동반이 필요하며, 휠체어 대여는 제공되지 않습니다
  - [주변] 주변 지역에 안전사고 이력이 있습니다

■ 김녕요트투어
  - 유형: 액티비티
  - 접근성: 완전 이용 가능
  - 시설 이용 가능 / 활동 참여 미확인
  - [확인필요] 활동 참여 가능 여부는 확인되지 않았습니다. 보조 인력이 필요할 수 있으니 시설에 미리 문의해 주세요
  - 장애인화장실 있음
  - [주의] 현장 확인된 등급이 아닙니다. 유사 시설에서 탕·탑승 구간에 계단이 있는 사례가 확인되어, 방문 전 시설에 직접 문의를 권합니다
  - [주변] 주변 지역에 안전사고 이력이 있습니다

■ 종달리해안도로
  - 유형: 트레일·산책로
  - 접근성: 부분 이용 가능
  - 시설 이용 가능 / 활동 참여 불가
  - [주의] 장애인화장실 미확인 — 장시간 체류 시 사전 확인 필요
  - [주변] 주변 지역에 안전사고 이력이 있습니다

■ 용두암해수랜드
  - 유형: 웰니스
  - 접근성: 완전 이용 가능
  - 시설 이용 가능 / 활동 참여 미확인
  - [확인필요] 활동 참여 가능 여부는 확인되지 않았습니다. 보조 인력이 필요할 수 있으니 시설에 미리 문의해 주세요
  - [주의] 장애인화장실 미확인 — 장시간 체류 시 사전 확인 필요 / 현장 확인된 등급이 아닙니다. 유사 시설에서 탕·탑승 구간에 계단이 있는 사례가 확인되어, 방문 전 시설에 직접 문의를 권합니다
  - [주변] 주변 지역에 안전사고가 잦은 편입니다. 이동 중 주의해 주세요"""
VERBATIM_C = ["탕 내부에 계단이 있어 휠체어 단독 이용은 위험합니다. 보호자 동반이 필요하며, 휠체어 대여는 제공되지 않습니다", "현장 확인된 등급이 아닙니다. 유사 시설에서 탕·탑승 구간에 계단이 있는 사례가 확인되어, 방문 전 시설에 직접 문의를 권합니다", "장애인화장실 미확인 — 장시간 체류 시 사전 확인 필요", "장애인화장실 미확인 — 장시간 체류 시 사전 확인 필요 / 현장 확인된 등급이 아닙니다. 유사 시설에서 탕·탑승 구간에 계단이 있는 사례가 확인되어, 방문 전 시설에 직접 문의를 권합니다"]

ans_c, iss_c = run("휠체어로 온천이나 체험 활동 할 수 있어?",
                   CONTEXT_C, VERBATIM_C, "CASE C")

In [ ]:
# ============================================================
#  [대조군] 컨텍스트 없이 같은 질문 — KG 주입 효과 확인
#  소형 모델이 KG 없이 답하면 어떻게 되는지 비교용
# ============================================================
r = requests.post(URL, headers=HEADERS, timeout=300,
    json={"model":"jeju-tourism-agent",
          "input":"제주에서 물놀이하기 좋은 바다 알려줘","stream":False})
print(extract(r.json()))
print()
print("※ 위 답변에 어항 물놀이 금지, 안전요원 근무시간 안내가 없다면")
print("   KG 주입이 실질적인 안전 정보를 더한다는 근거가 됩니다.")

In [ ]:
# ============================================================
#  결과 요약
# ============================================================
for label, iss in [("CASE A", iss_a), ("CASE B", iss_b), ("CASE C", iss_c)]:
    print(f"{label}: {'통과' if not iss else '실패 ' + str(len(iss)) + '건'}")
    for i in iss: print("    -", i)
print()
print("실패가 있으면 → 프롬프트 규칙 강화 또는 응답 후 KG 원문 대체 필요")
print("전부 통과하면 → 현재 프롬프트 설계로 진행 가능")